# 02 Pipeline

This workflow is optimized for Lakehouse to Lakehouse data movement using PySpark so pipelines can make use of distributed and parallel processing.

FabricOps also supports reading from and writing to Fabric Warehouses when the pipeline requires Warehouse-based sources or targets.

### `read_lakehouse_table`

Use this for reading a complete Delta table from a configured Fabric Lakehouse.

### `read_warehouse_table`

Use this for reading a complete table from a configured Fabric Warehouse.

### `read_warehouse_query`

Use this when the source must be read through a SQL query. The example uses `SELECT * FROM schema.table`, so the returned DataFrame still represents the complete physical source table.

When working with a very large Warehouse table, edit the SQL query to select only the required columns or rows. A filtered, joined or aggregated query result should not be registered as the complete profile of a single physical source table.

`profile_and_register_table` profiles the DataFrame representing a physical source or target table and writes its observed structure, statistics, catalogue identity and lineage participation into the FabricOps metadata tables.

The table identity is resolved from the active environment, configured target, schema and table name. Run this function after successfully reading a source table or after successfully writing and reading a target table.

## 1. Run `00_env_config`

Run the `00_env_config` notebook before continuing. It establishes the active environment, configured Fabric stores, schemas and runtime context used by the pipeline.

In [ ]:
%run 00_env_config


## 2. Import required functions

In [ ]:
from fabricops_kit import (
    profile_and_register_table,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    widget_view_data_contract,
    write_lakehouse_table,
    write_warehouse_table,
)


## 3. Read and profile source tables

Choose the example that matches your source. Define the target, schema and table name once, then reuse the same values for both reading and profiling. Run only one of the following examples.

### Example A: Lakehouse source

In [ ]:
SOURCE_TARGET = "unified"
SOURCE_SCHEMA = UNIFIED_SCHEMA
SOURCE_TABLE_NAME = "smoke_test_source_df"

source_df = read_lakehouse_table(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    spark_session=spark,
)

source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


### Example B: Warehouse source

In [ ]:
SOURCE_TARGET = "product"
SOURCE_SCHEMA = PRODUCT_SCHEMA
SOURCE_TABLE_NAME = "smoke_test_source_df"

source_df = read_warehouse_table(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    spark_session=spark,
)

source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


### Example C: Warehouse SQL query source

In [ ]:
SOURCE_TARGET = "product"
SOURCE_SCHEMA = PRODUCT_SCHEMA
SOURCE_TABLE_NAME = "smoke_test_source_df"
SOURCE_QUERY = f"SELECT * FROM {SOURCE_SCHEMA}.{SOURCE_TABLE_NAME}"

source_df = read_warehouse_query(
    SOURCE_QUERY,
    target=SOURCE_TARGET,
    spark_session=spark,
)

# This profile is valid because the query returns the complete physical table.
source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


## 4. Source guardrails

Source tables are checked against guardrails set by Governance before processing continues. Depending on the configured rule, FabricOps will warn the user or stop the pipeline when a source does not meet the approved requirement.

In [ ]:
# Planned for v0.3.0


## 5. User defined transformation

Perform the transformations required by your pipeline. Use Microsoft Fabric Copilot to assist with transformation logic where relevant.

## 6. Target guardrails

Target tables are checked against guardrails set by Governance before publication. Depending on the configured rule, FabricOps will warn the user or stop the pipeline when a target does not meet the approved requirement.

In [ ]:
# Planned for v0.3.0


## 7. Write and profile the target table

Write the transformed DataFrame to its configured Fabric target. After the write succeeds, read the persisted table and profile the physical target that now exists.

In [ ]:
TARGET_TARGET = "unified"
TARGET_SCHEMA = UNIFIED_SCHEMA
TARGET_TABLE_NAME = "smoke_test_target_df"

write_lakehouse_table(
    transformed_df,
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
)

target_df = read_lakehouse_table(
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
    spark_session=spark,
)

target_profile_df = profile_and_register_table(
    target_df,
    profile_role="target",
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
)


### Optional Warehouse target

In [ ]:
TARGET_TARGET = "product"
TARGET_SCHEMA = PRODUCT_SCHEMA
TARGET_TABLE_NAME = "smoke_test_target_df"

write_warehouse_table(
    transformed_df,
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
)

target_df = read_warehouse_table(
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
    spark_session=spark,
)

target_profile_df = profile_and_register_table(
    target_df,
    profile_role="target",
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
)


## 8. View the resulting data contract

Open the data contract widget to review the registered source and target metadata produced by this pipeline.

In [ ]:
widget_view_data_contract(spark_session=spark)
